# 04 · Modelo de Fusión y Ablation Study
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Isaac Oviedo  
> **Objetivo:** Combinar CNN 1D + RF en un clasificador de fusión y demostrar con ablation study que la combinación supera a cada modelo individual.

---

In [ ]:
import sys
sys.path.insert(0, "..")

import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)

from src.preprocessing import preprocess
from src.models import (
    get_cnn_representation, train_fusion,
    evaluate_model, build_cnn_model, build_rf
)
from src.config import CLASS_LABELS

UR_RED = "#DA0921"; UR_NAVY = "#242839"; UR_TECH = "#0E6A8C"; UR_GREEN = "#1A6E3A"

data = preprocess("../data/evaluaciones_docentes.csv")

# Cargar modelos entrenados
cnn_model = tf.keras.models.load_model("../models/cnn_model.keras")
with open("../models/rf_model.pkl", "rb") as f:
    rf_model = pickle.load(f)
print("✅ Modelos cargados")

## 1 · Extraer representaciones

In [ ]:
# CNN: vector de representación textual (64d)
repr_train = get_cnn_representation(cnn_model, data.X_text_train)
repr_test  = get_cnn_representation(cnn_model, data.X_text_test)

# RF: probabilidades de clase (3d)
rf_proba_train = rf_model.predict_proba(data.X_num_train)
rf_proba_test  = rf_model.predict_proba(data.X_num_test)

print(f"CNN repr train: {repr_train.shape}  (64 dimensiones semánticas)")
print(f"RF proba train: {rf_proba_train.shape}  (3 probabilidades de clase)")
print(f"Vector de fusión: {repr_train.shape[1] + rf_proba_train.shape[1]}d")

## 2 · Entrenar el modelo de fusión

In [ ]:
fusion_model, fusion_history = train_fusion(
    X_repr_train=repr_train,
    X_proba_rf_train=rf_proba_train,
    y_train=data.y_train,
    X_repr_val=repr_test,
    X_proba_rf_val=rf_proba_test,
    y_val=data.y_test,
)
fusion_model.summary()

In [ ]:
# Curvas de aprendizaje del modelo de fusión
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (train_m, val_m), title in zip(
    axes,
    [("accuracy", "val_accuracy"), ("loss", "val_loss")],
    ["Accuracy", "Loss"],
):
    epochs = range(1, len(fusion_history[train_m]) + 1)
    ax.plot(epochs, fusion_history[train_m], color=UR_TECH, linewidth=2, label="Train")
    if val_m in fusion_history:
        ax.plot(epochs, fusion_history[val_m], color=UR_RED, linewidth=2,
                linestyle="--", label="Validación")
    ax.set_title(f"Fusión - {title}", fontsize=12, fontweight="bold", color=UR_NAVY)
    ax.legend(frameon=False)

plt.suptitle("Curvas de aprendizaje - Modelo de Fusión",
             fontsize=14, fontweight="bold", color=UR_RED, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/fusion_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 3 · Ablation study

In [ ]:
# CNN solo
cnn_proba_test = cnn_model.predict(data.X_text_test, verbose=0)
cnn_pred_test  = np.argmax(cnn_proba_test, axis=1)
metrics_cnn = evaluate_model(data.y_test, cnn_pred_test, cnn_proba_test, "CNN 1D (solo texto)")

# RF solo
rf_pred_test   = rf_model.predict(data.X_num_test)
rf_proba_test  = rf_model.predict_proba(data.X_num_test)
metrics_rf  = evaluate_model(data.y_test, rf_pred_test, rf_proba_test, "Random Forest (solo numérico)")

# Fusión
X_fusion_test      = np.concatenate([repr_test, rf_proba_test], axis=1)
fusion_proba_test  = fusion_model.predict(X_fusion_test, verbose=0)
fusion_pred_test   = np.argmax(fusion_proba_test, axis=1)
metrics_fusion = evaluate_model(data.y_test, fusion_pred_test, fusion_proba_test, "Fusión CNN + RF")

print("=" * 65)
print(f"  {'Modelo':<35} {'Accuracy':>9} {'F1-macro':>9} {'AUC-ROC':>9}")
print("  " + "-" * 62)
for m in [metrics_cnn, metrics_rf, metrics_fusion]:
    marker = " ✅" if m["model_name"] == "Fusión CNN + RF" else ""
    print(f"  {m['model_name']:<35} {m['accuracy']:>9.4f} {m['f1_macro']:>9.4f} {m['auc_roc_macro']:>9.4f}{marker}")
print("=" * 65)

delta_f1 = metrics_fusion["f1_macro"] - metrics_rf["f1_macro"]
print(f"\n  Δ F1-macro fusión vs RF solo: +{delta_f1:.4f} ({delta_f1*100:.2f} puntos porcentuales)")

In [ ]:
# Visualización del ablation study
fig, ax = plt.subplots(figsize=(11, 5))
models_names = ["CNN 1D\n(solo texto)", "Random Forest\n(solo numérico)", "Fusión\nCNN + RF"]
accs = [metrics_cnn["accuracy"], metrics_rf["accuracy"], metrics_fusion["accuracy"]]
f1s  = [metrics_cnn["f1_macro"], metrics_rf["f1_macro"], metrics_fusion["f1_macro"]]
aucs = [metrics_cnn["auc_roc_macro"], metrics_rf["auc_roc_macro"], metrics_fusion["auc_roc_macro"]]

x = np.arange(len(models_names)); w = 0.25
b1 = ax.bar(x - w, accs, w, label="Accuracy", color=UR_TECH, alpha=0.9, edgecolor="white")
b2 = ax.bar(x,     f1s,  w, label="F1-macro", color=UR_NAVY, alpha=0.9, edgecolor="white")
b3 = ax.bar(x + w, aucs, w, label="AUC-ROC",  color=UR_RED,  alpha=0.9, edgecolor="white")

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=8.5, color=UR_NAVY)

ax.set_xticks(x); ax.set_xticklabels(models_names)
ax.set_ylim(0.3, 1.05)
ax.set_title("Ablation Study - Comparación de modelos", fontsize=13, fontweight="bold", color=UR_NAVY)
ax.legend(frameon=False)
ax.axvline(1.5, color="#CCCCCC", linewidth=1, linestyle="--")
ax.text(1.6, 0.33, "-> Fusión", color=UR_GREEN, fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · SHAP - Explicabilidad del Random Forest

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(data.X_num_test[:200])

    from src.config import NUMERIC_FEATURES
    fig, ax = plt.subplots(figsize=(10, 5))
    # Importancia SHAP (media absoluta por feature y clase)
    shap_importance = np.abs(np.array(shap_values)).mean(axis=(0, 2)) if isinstance(shap_values, list)         else np.abs(shap_values).mean(axis=0)
    if isinstance(shap_values, list):
        shap_importance = np.abs(np.array(shap_values)).mean(axis=(0, 2))
    else:
        shap_importance = np.abs(shap_values).mean(axis=0)

    sorted_idx = np.argsort(shap_importance)
    ax.barh([NUMERIC_FEATURES[i] for i in sorted_idx],
            shap_importance[sorted_idx],
            color=UR_RED, alpha=0.85, edgecolor="white")
    ax.set_title("Importancia SHAP - features más influyentes",
                 fontsize=13, fontweight="bold", color=UR_NAVY)
    ax.set_xlabel("SHAP medio absoluto")
    plt.tight_layout()
    plt.savefig("../outputs/shap_importance.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("✅ SHAP calculado exitosamente")
except ImportError:
    print("⚠️  shap no instalado. Ejecutar: pip install shap")

In [ ]:
# Guardar fusión
fusion_model.save("../models/fusion_model.keras")
print("✅ Modelo de fusión guardado en models/fusion_model.keras")